In [76]:
import pandas as pd
import geopandas as gpd
import json
from shapely.geometry import shape
from shapely import wkt
import warnings
from shapely.ops import linemerge, unary_union

# filter all warnings
warnings.filterwarnings("ignore")

In [77]:
neigh = pd.read_csv("Data/Neigh/neigh.csv")

neigh["geometry"] = neigh["geom"].apply(wkt.loads)

neigh = gpd.GeoDataFrame(
    neigh,
    geometry="geometry",
    crs="EPSG:4326"
)
neigh = neigh[["name", "geometry"]]
neigh["geometry"] = neigh.buffer(0)



In [78]:
with open('data/Bus/AMB.json') as f:
    data = json.load(f)


stops_amb = []
traj_amb = []

for collection in data:
    for feature in collection["features"]:
        props = feature["properties"]
        geom = feature["geometry"]
        if geom["type"] == "Point":
            stops_amb.append({
                "code": props["COD_EMT"],
                "name": props["NOMBRE"],
                "geometry": shape(geom)
            })
        if geom["type"] != "Point":
            traj_amb.append({
                "linia": props["LINEA"],
                "sentit": props["SENTIDO"],
                "geometry": shape(geom)
            })

geo_df_stops_amb = gpd.GeoDataFrame(stops_amb,geometry="geometry", crs="EPSG:4326")
geo_df_traj_amb = gpd.GeoDataFrame(traj_amb, geometry="geometry", crs="EPSG:4326")


buffered = neigh.copy()
buffered.to_crs('EPSG:25831', inplace=True)
buffered["geometry"] = buffered.geometry.buffer(2500)
geo_df_stops_amb.to_crs('EPSG:25831', inplace=True)
geo_df_stops_amb = gpd.sjoin(
    geo_df_stops_amb,
    buffered[["geometry"]],
    predicate="within",
    how="inner",
).drop(columns="index_right")
geo_df_stops_amb.to_crs('EPSG:4326', inplace=True)

geo_df_traj_amb = (geo_df_traj_amb.groupby("linia", as_index=False).agg(geometry=("geometry", lambda s: unary_union(list(s)))))

geo_df_traj_amb = gpd.GeoDataFrame(geo_df_traj_amb,geometry="geometry",crs="EPSG:4326")

boundary = buffered.union_all()
geo_df_traj_amb.to_crs('EPSG:25831', inplace=True)
geo_df_traj_amb["inside_ratio"] = (
    geo_df_traj_amb.geometry.intersection(boundary).length
    / geo_df_traj_amb.geometry.length
)

geo_df_traj_amb = geo_df_traj_amb[
    geo_df_traj_amb["inside_ratio"] >= 0.60   # keep lines with >=20% inside
]


geo_df_traj_amb.to_crs('EPSG:4326', inplace=True)
geo_df_traj_amb.drop_duplicates(subset=['linia'],inplace=True)
geo_df_stops_amb.drop_duplicates(subset=['code'],inplace=True)

In [79]:
# delete all the zeros before the bus stop code
geo_df_stops_amb["code"] = geo_df_stops_amb["code"].apply(lambda x: x.lstrip("0")).astype(int)
geo_df_stops_amb

,code,name,geometry
223,200103,Riera Bonet,POINT (2.02769 41.40477)
267,200152,Esglesia St. Bartomeu,POINT (2.03807 41.42635)
268,200178,Esglesia St. Bartomeu,POINT (2.03821 41.42648)
277,200153,Ctra. de Vallvidrera,POINT (2.04046 41.42781)
300,105724,Roserar Dot i Camprubi,POINT (2.04579 41.39055)
...,...,...,...
12809,107905,Poliesportiu Marina Besos,POINT (2.23174 41.42347)
12813,102334,Salvador Espriu,POINT (2.23246 41.44564)
12815,110367,Tortosa - Guifre,POINT (2.23514 41.43701)
12823,107908,La Mora,POINT (2.23756 41.43223)


In [80]:
bus_stops_w_line = pd.read_excel("Data/Bus/all_lines.xlsx")
bus_stops_w_line.rename(columns={"Código de parada / Codi de parada": "code",'Municipio / Municipi': 'municipi','Líneas / Línies': 'lines'}, inplace=True)
bus_stops_w_line = bus_stops_w_line[["code", "municipi", "lines"]]
geo_df_stops_amb = geo_df_stops_amb.merge(bus_stops_w_line, on="code", how="left")
geo_df_stops_amb = geo_df_stops_amb[geo_df_stops_amb["lines"].notnull()]
geo_df_stops_amb["lines"] = geo_df_stops_amb["lines"].apply(lambda x: [line.strip() for line in x.split("-")])
geo_df_stops_amb = geo_df_stops_amb.explode("lines", ignore_index=True)
# delete lines that start with N
geo_df_stops_amb = geo_df_stops_amb[~geo_df_stops_amb["lines"].str.startswith("N")]
geo_df_stops_amb['lines'].nunique(), geo_df_stops_amb['code'].nunique()

(187, 3137)

In [81]:
linies_w_trajectory = geo_df_traj_amb["linia"].unique().tolist()
geo_df_traj_amb["linia"].nunique()

156

In [82]:
geo_df_stops_amb = geo_df_stops_amb[geo_df_stops_amb["lines"].isin(linies_w_trajectory)]

In [83]:
geo_df_stops_amb['lines'].nunique(), geo_df_stops_amb['code'].nunique()

(132, 2939)

In [86]:
geo_df_stops_amb[geo_df_stops_amb['lines'] == 'V33'].sort_values(by='name', ascending=True)

,code,name,geometry,municipi,lines
5021,3598,Alfons el Magnanim - Jaume Huguet,POINT (2.21249 41.41786),Barcelona,V33
5022,3597,Alfons el Magnanim - Jaume Huguet,POINT (2.21258 41.41805),Barcelona,V33
5026,2660,Argentina - Bernat Metge,POINT (2.21263 41.42129),Sant Adrià de Besòs,V33
3259,2577,Argentina - Bernat Metge,POINT (2.21275 41.42147),Sant Adrià de Besòs,V33
3217,84,Av Santa Coloma,POINT (2.20788 41.44818),Santa Coloma de Gramenet,V33
7097,729,Balmes - Alarcon,POINT (2.21087 41.42963),Sant Adrià de Besòs,V33
7070,728,Balmes - Almirall Oquendo,POINT (2.20774 41.43115),Sant Adrià de Besòs,V33
3266,967,Balmes - Jovellanos,POINT (2.21032 41.43012),Sant Adrià de Besòs,V33
8366,968,Balmes - Pereda,POINT (2.20789 41.43133),Sant Adrià de Besòs,V33
3331,666,Banus - Metro Santa Rosa,POINT (2.21405 41.44632),Santa Coloma de Gramenet,V33
